In [1]:
import os
import glob
import sys

import ROOT
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tabulate import tabulate
from yaml import safe_load, YAMLError
import pickle
import config as cfg 
from efficiency_tools import efficiency_finder
import basic_functions
from df_makers import data_to_pickle_function
from apply_bdt_and_pickle_df import load_bdt_and_apply

ROOT.DisableImplicitMT() 



Welcome to JupyROOT 6.28/10


Warning in <ROOT_TImplicitMT_DisableImplicitMT>: Implicit multi-threading is already disabled


In [9]:
sq_BDT_cut = 0.99965
samples = ["p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu"]
chunk = 3

#path to data and outputs
inputpath    = basic_functions.check_inputpath(f"/r02/lhcb/ejnw2/fcc_2025/FCCAnalyses/examples/FCCee/flavour/B2Inv/outputs/Bu2lnu_background_no_lepton_veto/root_bdtscores/bdtlh_nocut/{samples[0]}/chunk_{chunk}_with_evtid.root")
yamlpath     = basic_functions.check_inputpath(cfg.fccana_opts['yamlPath'])

In [10]:
#Get list of vars to save
bdtvars_list_optimised = cfg.optimised_bdt_lh_opts["mvaBranchList"]
responsevars = ["EVT_hemisEmin_Emiss"] 

bdtvars      = list(set(basic_functions.vars_fromyaml(yamlpath, bdtvars_list_optimised)))
flavtag_vars = list(basic_functions.vars_fromyaml(yamlpath, "flavour-tag-vars"))
truth_vars = list(basic_functions.vars_fromyaml(yamlpath,"MCtruth-vars"))

vars_to_save = list(set(bdtvars+truth_vars+["EVT_hemisEmin_nLept"]+ responsevars))

vars_to_save = vars_to_save + ["evt_id"] + ["chunk"]

In [11]:
# list of inputs to say which BDT to use
BDT_params = {"config_bdtopts": cfg.optimised_bdt_lh_opts,
                   "training_round": "baseline-plus-hps",
                   "hps_dict_name":"baseline-plus-hps",
                   "features_list_name": "bdtlh-vars-v1",
                   "bdt_label": "_lh"}




In [12]:
cut = "EVT_hemisEmin_nLept == 0"

#apply BDT and cut

eff_bdtlh_cut = {}
N_before_bdtlh_cut = {}

# now collect relevant events into a dataframe
decay = samples[0]
print(f'Starting processing decay: {decay}')
N_pre=0
N_post=0

nchunks = 1
chunked_populated_files = [inputpath]


for n in range(nchunks):
    print(f'---> Starting processing chunk {n} of {nchunks}')
    files =chunked_populated_files[n]
    Rdf = ROOT.RDataFrame("events", files)
    if cut:
        Rdf = Rdf.Filter(cut)
    Rdf_np = Rdf.AsNumpy(columns= vars_to_save)
    sub_df = pd.DataFrame(Rdf_np)
    sub_df["decay"] = decay

    # want to make sure that integer types are actually set as integers - currenlty stored as float
    #if changed branches significantly might be worth checking the list is still right, with current branches expected integers in yaml
    integer_branches = [s for s in vars_to_save if '_n' in s and '_norm' not in s]
    for integer_branch in integer_branches:
        sub_df[integer_branch] = sub_df[integer_branch].astype(np.int32)

    #####################
    # apply BDT and cut #
    #####################
    
    model, bdtname, dataframe_chunk = load_bdt_and_apply(sub_df, 
                        config_bdtopts = BDT_params["config_bdtopts"],
                        training_round = BDT_params["training_round"],
                        hps_dict_name = BDT_params["hps_dict_name"],
                        features_list_name = BDT_params["features_list_name"],
                        bdt_label = BDT_params["bdt_label"])
        

    N_pre += len(dataframe_chunk)
    cut_df_chunk = dataframe_chunk[((1-dataframe_chunk['bdt_score_1'])>sq_BDT_cut)&((1-dataframe_chunk['bdt_score_0'])>sq_BDT_cut)]
    N_post += len(cut_df_chunk)


    N_before_bdtlh_cut[decay] = N_pre 
    eff_bdtlh_cut[decay] = N_post/N_pre 
    

print(len(cut_df_chunk[((1-cut_df_chunk["bdt_score_0"])>sq_BDT_cut)&((1-cut_df_chunk["bdt_score_1"])>sq_BDT_cut)]))


Starting processing decay: p8_ee_Zbb_ecm91_EvtGen_Bu2TauNuTau2MuNuNu
---> Starting processing chunk 0 of 1
Loading BDT model...
BDT model loaded successfully.
146


In [14]:
### Open saved ROOT file:
root_file = f"/r02/lhcb/ejnw2/fcc_2025/FCCAnalyses/examples/FCCee/flavour/B2Inv/outputs/Bu2lnu_background_no_lepton_veto/root_bdtscores/bdtlh_nocut/{samples[0]}/chunk_{chunk}_with_bdtcut0-99965.root"
root_Rdf = ROOT.RDataFrame("events", root_file)
root_Rdf_np = root_Rdf.AsNumpy(columns= vars_to_save+["bdt_score_0","bdt_score_1","bdt_score_2"])
root_sub_df = pd.DataFrame(root_Rdf_np)

In [15]:
print(len(root_sub_df[((1-root_sub_df["bdt_score_0"])>sq_BDT_cut)&((1-root_sub_df["bdt_score_1"])>sq_BDT_cut)]))

146
